<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 2.6: Testing with ChiselSim
**Prev: [Putting it all Together: An FIR Filter](2.5_exercise.ipynb)**<br>
**Next: [Generators: Parameters](3.1_parameters.ipynb)**

## Motivation
So far we have used `simulate(...)` to test our circuits without saying much about how it works.
`simulate` is Chisel's built-in simulation and testing library, **ChiselSim** (package `chisel3.simulator`).
It replaces the older `chisel-testers`/`iotesters` and the separate `chiseltest` library, which were used with earlier versions of Chisel.
ChiselSim provides the following benefits:

- it is part of Chisel itself, so there is no extra dependency to manage
- it is suitable for both unit tests and system integration tests
- it is designed for composable abstractions and layering: a test is just ordinary Scala code, so you can build your own helper functions out of the primitives
- it compiles your design to SystemVerilog and runs it on a real simulator (Verilator in this bootcamp), so what you simulate is what you would synthesize
- it works well with unit-test frameworks like ScalaTest (see `chisel3.simulator.scalatest.ChiselSim`) when you move from a notebook to a real project

In this module we review the basic operations (`poke`, `peek`, `expect` and `step`), and then learn how to test modules with `Decoupled` (ready/valid) interfaces by writing small helper functions ourselves.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.experimental.BundleLiterals._
import chisel3.simulator.PeekPokeAPI._

>This bootcamp requires some slight differences from the imports you might see 
elsewhere for Chisel. The `import chisel3.simulator.PeekPokeAPI._` brings in the 
`poke`, `peek`, `expect` and `step` methods on Chisel signals, and the `simulate(...)` function
used below is a thin wrapper (defined in `source/load-ivy.sc`) around ChiselSim's `EphemeralSimulator.simulate`
that prints failures nicely in the notebook.

---
# Basic Tester implementation

ChiselSim starts with the same basic operations as the older `iotesters`. Here's a brief summary of the basic
functionality mapping between the older `iotesters` and ChiselSim:

|        | iotesters             | ChiselSim            |
| :----  | :---                  | :---                |
| poke   | poke(c.io.in1, 6)     | c.io.in1.poke(6.U)    |
| peek   | peek(c.io.out1)       | c.io.out1.peek()      |
| expect | expect(c.io.out1, 6)  | c.io.out1.expect(6.U) |
| step   | step(1)               | c.clock.step(1)  |
| initiate | Driver.execute(...) { c => | simulate(...) { c => |


Let's start by looking at the simple pass through module from 2.1

In [ ]:
// Chisel Code, but pass in a parameter to set widths of ports
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

Using the old `iotesters` style (which no longer exists in current Chisel) a simple test would look like this:

```scala
val testResult = Driver(() => new Passthrough()) {
  c => new PeekPokeTester(c) {
    poke(c.io.in, 0)     // Set our input to value 0
    expect(c.io.out, 0)  // Assert that the output correctly has 0
    poke(c.io.in, 1)     // Set our input to value 1
    expect(c.io.out, 1)  // Assert that the output correctly has 1
    poke(c.io.in, 2)     // Set our input to value 2
    expect(c.io.out, 2)  // Assert that the output correctly has 2
  }
}
assert(testResult)   // Scala Code: if testResult == false, will throw an error
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!
```

With ChiselSim the same test is written as follows.


In [ ]:
simulate(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

>Just to illustrate the way ChiselSim advances the clock we can
add some `step` operations to the previous example.

In [ ]:
simulate(new PassthroughGenerator(16)) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.clock.step(1)    // advance the clock
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.clock.step(1)    // advance the clock
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.clock.step(1)    // advance the clock
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}

---
## What to notice in the above example

ChiselSim's `simulate` method requires very little boilerplate: there is no separate tester class, the
test body is just a Scala function that receives the module under test (`c`).

The `poke`, `peek` and `expect` methods are part of each individual `io` element.
This gives important hints to the tester to make better checking of types.
The clock is also a signal of the module (`c.clock`), so `step` is a method on it: `c.clock.step(n)`.

Another difference is that values poked and expected are Chisel literals.
Although pretty simple here, it also provides stronger checking in more advanced and interesting examples, including `Bundle` literals, which we will see below.

Three more things to be aware of:
- `simulate` first applies a reset to the module (one clock cycle with `reset` high), so `RegInit` registers start at their reset values before your test body runs. Registers without an initial value (plain `Reg`) hold random values until written.
- `peek()` returns a Chisel literal, so use `.litValue` to get a Scala `BigInt` (for example `c.io.out.peek().litValue`), and `peekBoolean()` on a `Bool` to get a Scala `Boolean`.
- Like `poke` and `expect`, a failing `expect` simply throws a Scala exception, which is how the test fails.


# Modules with Decoupled Interfaces
In this section we will look at how to test modules with `Decoupled` interfaces.
`Decoupled` takes a chisel data type and provides it with `ready` and `valid` signals.
ChiselSim has no special support for these interfaces, but we do not need any: a test is ordinary Scala code, so we can write small helper functions that drive `ready` and `valid` for us, and reuse them in every test of every `Decoupled` circuit.

## A queue example
The `QueueModule` passes through data whose type is determined by `ioType`. There are `entries` state elements inside the `QueueModule` meaning it can hold that many elements before it exerts backpressure.

In [ ]:
class QueueModule[T <: Data](ioType: T, entries: Int) extends Module {
  val in = IO(Flipped(Decoupled(ioType)))
  val out = IO(Decoupled(ioType))
  out <> Queue(in, entries)
}

In [ ]:
// Tests are plain Scala, so we can write our own helpers.

// Add (enqueue) one element to a Decoupled input interface.
// `drive` pokes the payload, e.g. `_.poke(5.U)`.
// We hold `valid` high until the DUT is `ready`; the transfer happens on the clock edge.
def enqueue[T <: Data](port: DecoupledIO[T], clock: Clock, timeout: Int = 1000)(drive: T => Unit): Unit = {
  port.valid.poke(true.B)
  drive(port.bits)
  var cycles = 0
  while (!port.ready.peekBoolean()) {
    clock.step(1)
    cycles += 1
    assert(cycles < timeout, s"timeout: ready was not asserted after $timeout cycles")
  }
  clock.step(1)             // the handshake (valid && ready) completes on this edge
  port.valid.poke(false.B)
}

// Remove (dequeue) one element from a Decoupled output interface and check it.
// `check` examines the payload, e.g. `_.expect(5.U)`.
// We raise `ready` and wait until the DUT presents a valid element.
def expectDequeue[T <: Data](port: DecoupledIO[T], clock: Clock, timeout: Int = 1000)(check: T => Unit): Unit = {
  port.ready.poke(true.B)
  var cycles = 0
  while (!port.valid.peekBoolean()) {
    clock.step(1)
    cycles += 1
    assert(cycles < timeout, s"timeout: valid was not asserted after $timeout cycles")
  }
  check(port.bits)
  clock.step(1)             // the handshake completes on this edge
  port.ready.poke(false.B)
}

## Enqueue and dequeue helpers
Before running the test body, `simulate` resets the module for one clock cycle (this is what initializes the `Queue`'s internal state). If a test ever needs to restart the circuit, drive the reset like any other input: `c.reset.poke(true.B); c.clock.step(1); c.reset.poke(false.B)`.


In this example we define our own helper functions to insert and extract values from the `queue`. 

| function | description |
| :---   | :---        |
| enqueue | Adds (enqueues) one element to a `Decoupled` input interface, waiting until the DUT is `ready` |
| expectDequeue | Removes (dequeues) one element from a `Decoupled` output interface, waiting until it is `valid`, and checks its value |
---

The functions are generic in the payload type `T <: Data`. The last argument is a function that does the type-specific part: poking or checking the `bits` field. 
The `ready`/`valid` handshake itself is the same for every `Decoupled` interface, so we write it only once.

>Note: Remember that a transfer happens on a clock edge where both `valid` and `ready` are high. The helpers therefore keep `valid` (or `ready`) high until the handshake can happen, step the clock once, and then lower the signal again. The `timeout` makes a hung handshake fail the test instead of looping forever.


In [ ]:
simulate(new QueueModule(UInt(9.W), entries = 200)) { c =>
    c.out.ready.poke(false.B)
    c.in.valid.poke(false.B)

    val testVector = Seq.tabulate(200){ i => i.U }

    testVector.foreach { value =>
      enqueue(c.in, c.clock)(_.poke(value))
      expectDequeue(c.out, c.clock)(_.expect(value))
    }
}
println("SUCCESS!!")

## Enqueuing and dequeuing whole sequences
Because a test is just Scala, enqueuing or dequeuing a whole sequence is just a `foreach` over the helper functions we wrote above.

| operation | description |
| :---   | :---        |
| `testVector.foreach(v => enqueue(c.in, c.clock)(_.poke(v)))` | Continues to add (enqueue) elements from the `Seq` to a `Decoupled` input interface, one at a time, until the sequence is exhausted |
| `testVector.foreach(v => expectDequeue(c.out, c.clock)(_.expect(v)))` | Removes (dequeues) elements from a `Decoupled` output interface, one at a time, and compares each one to the next element of the `Seq` |
---
> Note: The example below works fine but, as written, all the enqueues must finish before the dequeues can begin. This example would fail if the `testVector`'s size is made larger than the queue depth, because the queue would fill up and `enqueue` could never complete (our `timeout` assertion reports this). Try it yourself to see what the failure looks like. In the next section we will show how to construct this type of test properly.


In [ ]:
simulate(new QueueModule(UInt(9.W), entries = 200)) { c =>
    c.out.ready.poke(false.B)
    c.in.valid.poke(false.B)

    val testVector = Seq.tabulate(100){ i => i.U }

    testVector.foreach { v => enqueue(c.in, c.clock)(_.poke(v)) }
    testVector.foreach { v => expectDequeue(c.out, c.clock)(_.expect(v)) }
}
println("SUCCESS!!")

> One important take away from the last section is that the functions we just saw, `enqueue` and `expectDequeue`, are not complicated special case logic in ChiselSim.
Rather they are examples of the kinds of harness building that ChiselSim encourages you to do on top of its primitives (`poke`, `peek`, `expect` and `step`). Building up your own library of such helpers is a good way to keep your tests short and readable.

# Driving Several Interfaces at Once

Sometimes sending and receiving must happen at the same time. For example, a queue must be emptied while it is being filled if we want to push more elements through it than it can hold.
Some other testing libraries provide `fork` and `join` to run several test threads concurrently. ChiselSim's peek/poke API does not have them: the test is a single sequential Scala program.

That is not a limitation in practice. Hardware is synchronous, so all the "concurrency" is between clock edges. We simply write **one loop that runs once per clock cycle** and, in each iteration, drives the inputs of *every* interface, samples the handshakes, and then calls `step(1)`.

In the example below, the producer side offers the next element while there are elements left to send, and the consumer side is always ready. Each cycle we check whether a handshake (`valid && ready`) is happening on each side *before* the clock edge, and update our bookkeeping accordingly.
The loop is bounded by a cycle limit so that a bug cannot hang the test.

>In this style there is nothing hidden: you decide the order in which signals are poked and sampled in each cycle, so the behavior is completely deterministic.

In [ ]:
simulate(new QueueModule(UInt(9.W), entries = 200)) { c =>

    val n = 300                      // more elements than the queue can hold!
    var sent = 0
    var received = 0
    var cycles = 0

    c.out.ready.poke(true.B)         // the consumer is always ready

    while (received < n) {
      // producer: offer the next element while there are some left
      if (sent < n) {
        c.in.valid.poke(true.B)
        c.in.bits.poke(sent.U)
      } else {
        c.in.valid.poke(false.B)
      }

      // sample the handshakes that are about to happen on the next clock edge
      val enqFires = sent < n && c.in.ready.peekBoolean()
      val deqFires = c.out.valid.peekBoolean()    // out.ready is always true

      if (deqFires) {
        c.out.bits.expect(received.U)             // data must come out in order
        received += 1
      }
      if (enqFires) sent += 1

      c.clock.step(1)
      cycles += 1
      assert(cycles < 10 * n, "timeout")
    }
}
println("SUCCESS!!")

## Using Decoupled helpers with GCD
In this section we will use our helper functions to implement tests of *Greatest Common Divisor* **GCD**.
Let's start by defining our IO bundles. As you can see, there is no boilerplate needed to use `Bundle` *literals* (we imported `chisel3.experimental.BundleLiterals._` in the setup cell).

In [ ]:
class GcdInputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
}

In [ ]:
class GcdOutputBundle(val w: Int) extends Bundle {
  val value1 = UInt(w.W)
  val value2 = UInt(w.W)
  val gcd    = UInt(w.W)
}

Now let's look at a *Decoupled* version of **GCD**. We've modified it a bit here to use the `Decoupled` wrapper that adds a `ready` and a `valid` signal to the input and output Bundle. The `Flipped` wrapper takes the `Decoupled` `GcdInputBundle` which by default is created as an output and converts each field to the opposite direction (recursively). The data elements of the bundled arguments to `Decoupled` are placed in the top level field `bits`. 

In [ ]:
/**
  * Compute GCD using subtraction method.
  * Subtracts the smaller of registers x and y from the larger until register y is zero.
  * value input register x is then the Gcd
  * returns a packet of information with the two input values and their GCD
  */
class DecoupledGcd(width: Int) extends Module {

  val input = IO(Flipped(Decoupled(new GcdInputBundle(width))))
  val output = IO(Decoupled(new GcdOutputBundle(width)))

  val xInitial    = Reg(UInt())
  val yInitial    = Reg(UInt())
  val x           = Reg(UInt())
  val y           = Reg(UInt())
  val busy        = RegInit(false.B)

  input.ready := ! busy
  output.valid := false.B
  output.bits := DontCare

  when(busy)  {
    // during computation keep subtracting the smaller from the larger
    when(x > y) {
      x := x - y
    }.otherwise {
      y := y - x
    }
    when(x === 0.U || y === 0.U) {
      // when one of them becomes zero computation is over (the gcd is the other one),
      // signal valid data to output if the output is ready
      output.bits.value1 := xInitial
      output.bits.value2 := yInitial
      output.bits.gcd := x | y
      output.valid := true.B
      busy := ! output.ready
    }
  }.otherwise {
    when(input.valid) {
      // valid data available and no computation in progress, grab new values and start
      val bundle = input.deq()
      x := bundle.value1
      y := bundle.value2
      xInitial := bundle.value1
      yInitial := bundle.value2
      busy := true.B
    }
  }
}

Our test looks pretty much the same as the earlier Queue tests.
But there's more going on because the computation takes multiple cycles, so the input is not `ready` while each GCD is being computed.
The good news is that test side of this is simple and consistent across different Decoupled circuits: our `enqueue` and `expectDequeue` helpers wait for `ready` and `valid` for us.

We also introduce here the Chisel `Bundle` literal notation. Consider the line
```scala
new GcdInputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U)
```
`GcdInputBundle` defined above has two fields `value1` and `value2`.
We create a bundle literal by first creating a bundle then calling its `.Lit` method.
That method takes a variable argument list of key/value pairs, where the key (e.g. `_.value1`) is the field name and the value (e.g. x.U) is a chisel hardware literal, the Scala `Int` x is converted into a Chisel `UInt` literal.
The `_.` in front of the field name is necessary to bind the name value to the bundle internals. 
A whole bundle literal can be passed to `poke` and `expect`.

>This may not be the perfect notation but in extensive development discussions it was viewed as
the best balance between minimizing boilerplate and the notational limitations available in Scala.

The test below also waits 5 cycles before accepting each result (`c.output.ready` stays low during that time) to create backpressure on the GCD module.


In [ ]:
simulate(new DecoupledGcd(16)) { dut =>
  dut.input.valid.poke(false.B)
  dut.output.ready.poke(false.B)

  val testValues = for { x <- 1 to 10; y <- 1 to 10} yield (x, y)

  for ((x, y) <- testValues) {
    val input = new GcdInputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U)
    val expected = new GcdOutputBundle(16).Lit(_.value1 -> x.U, _.value2 -> y.U, _.gcd -> BigInt(x).gcd(BigInt(y)).U)

    enqueue(dut.input, dut.clock)(_.poke(input))
    dut.clock.step(5) // wait some cycles before receiving the output to create backpressure
    expectDequeue(dut.output, dut.clock)(_.expect(expected))
  }
}
println("SUCCESS!!")

---
# You're done!

[Return to the top.](#top)